## Khởi tạo Julia và tải dữ liệu benchmark

In [ ]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1

In [ ]:
!git clone https://github.com/cosude/Data-Mining-FIM-Group-7.git
%cd Data-Mining-FIM-Group-7
!julia --project=. src/download_benchmark.jl

## Chương 1: Cài đặt và kiểm thử

In [ ]:
!julia --project=. tests/runtests.jl

## Chương 2: Thực nghiệm và đánh giá

### 2.1. Kết quả đối chiếu với SPMF

In [ ]:
!julia --project=. experiments/exp1_correctness.jl

### 2.2. Mối quan hệ giữa minsupport và thời gian chạy

In [ ]:
!julia --project=. experiments/exp2_runtime_vs_minsup.jl

### 2.3. Mối quan hệ giữa số lượng tập phổ biến và minsupport

In [ ]:
!julia --project=. experiments/exp3_itemset_count.jl

### 2.4. So sánh hiệu năng bộ nhớ

In [ ]:
!julia --project=. experiments/exp4_memory.jl

### 2.5. Khả năng mở rộng đối với các mức kích thước cơ sở dữ liệu

In [ ]:
!julia --project=. experiments/exp5_scalability.jl

### 2.6. Ảnh hưởng của độ dài trung bình của giao dịch

In [ ]:
!julia --project=. experiments/exp6_avg_length.jl

## Chương 3: Ứng dụng thực tế

### 3.1. Ứng dụng 1: Phân tích Giỏ hàng

Để thực hiện phân tích giỏ hàng, tập dữ liệu được sử dụng là **Groceries** (dữ liệu bán lẻ thực tế) có định dạng CSV. Mỗi dòng tương ứng với một giao dịch, gồm các item biểu diễn dưới dạng chuỗi (VD: "banana"), ngăn cách nhau bởi dấu phẩy.

Do thuật toán FP-growth của nhóm cài đặt sử dụng cấu trúc dữ liệu có thuộc tính nhãn item được biểu diễn bởi số nguyên, kỹ thuật mã hóa các item từ **Groceries** sau được áp dụng:
1. Duyệt qua từng dòng của tập dữ liệu, lập chỉ mục ánh xạ một item thành một số nguyên duy nhất thông qua một bảng băm.
2. Chuyển đổi từng dòng giao dịch thành mảng số nguyên theo định dạng của SPMF.
3. Kết quả khai thác luật kết hợp sẽ chuyển đổi cách biểu diễn số nguyên về lại chuỗi tên item.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv"
urllib.request.urlretrieve(url, "groceries.csv")

item_to_id = {}
current_id = 1

with open("groceries.csv", "r", encoding="utf-8") as f, \
     open("encoded_groceries.txt", "w", encoding="utf-8") as data_out, \
     open("mapping.csv", "w", encoding="utf-8") as map_out:
    
    map_out.write("ID,Item\n")
    for line in f:
        items = [i.strip() for i in line.strip().split(',') if i.strip()]
        if not items: continue
        
        encoded_transaction = []
        for item in items:
            if item not in item_to_id:
                item_to_id[item] = current_id
                map_out.write(f"{current_id},{item}\n")
                current_id += 1
            encoded_transaction.append(str(item_to_id[item]))
            
        data_out.write(" ".join(encoded_transaction) + "\n")
print(f"Encoded sucessfully {len(item_to_id)} items.")

Đang xử lý và mã hóa tập dữ liệu Groceries...
-> Đã mã hóa thành công 9835 giao dịch.
-> Tổng số mặt hàng (items) duy nhất tìm thấy: 169

--- Ví dụ 3 giao dịch đầu tiên (Trước và sau mã hóa) ---
Gốc    : citrus fruit,semi-finished bread,margarine,ready soups
Mã hóa : 1 2 3 4

Gốc    : tropical fruit,yogurt,coffee
Mã hóa : 5 6 7

Gốc    : whole milk
Mã hóa : 8



In [ ]:
!julia --project=. src/app1_market_basket.jl